# ERA5 Vertical Profile Download for Tephigram Analysis

This script downloads ERA5 pressure-level meteorological data from the Copernicus Climate Data Store for a single point at Mace Head, Ireland. The output NetCDF file is intended for vertical-profile diagnostics and tephigram construction during the 14 February 2023 Saharan dust transport event.

## Purpose

The script retrieves vertical meteorological fields needed to build a tephigram and diagnose atmospheric structure above Mace Head. The downloaded variables are:

| Variable | ERA5 field | Unit | Tephigram use |
|---|---|---:|---|
| Temperature | `temperature` | K | Environmental temperature profile |
| Specific humidity | `specific_humidity` | kg kg⁻¹ | Dew-point and moisture profile |
| Zonal wind | `u_component_of_wind` | m s⁻¹ | Wind profile and shear |
| Meridional wind | `v_component_of_wind` | m s⁻¹ | Wind profile and shear |

The data are retrieved on pressure levels from 1000 to 100 hPa at 25 hPa spacing, allowing the lower and middle tropospheric structure to be examined in detail.

## Target event

The script downloads the ERA5 vertical profile for:

| Field | Value |
|---|---|
| Date | 14 February 2023 |
| Time | 16:00 UTC |
| Location | Mace Head region |
| Latitude | 53.25° N |
| Longitude | 9.50° W |
| Dataset | ERA5 pressure levels |
| Output format | NetCDF legacy |

The output file is:

```text
C:/Users/emman/OneDrive/Desktop/RCODES/era5_macehead_20230214_1600.nc



In [1]:
import os
import time
import random
from pathlib import Path
import cdsapi

CDS_KEYS = [
    "156656bf-4346-4228-8ea6-a47b8b807223",
    "1e0d795d-e02a-4ebc-81f9-0fb9c07f85a9",
    "1e912241-e383-49bc-a605-4f3155b60ad6",
    "9f9d3677-2705-4764-832b-c5d9bc917eb0",
    "9b7c19b2-d4c7-4ab6-9aef-33d57355862a",
    "2d7328f2-7e18-4fb4-ad55-2ca4605eb1c4",
    "22dec1aa-ecf2-4b87-9641-b80361fdba06",
    "d952344a-3e13-45a6-a7cd-4a7b8e220a79",
    "b6a8b25e-6d7f-46f8-80f6-cf0e37a2719b",
    "40b9f769-d2f6-4369-9b1f-15071345c640",
    "71c70660-fc78-41ce-836b-562ad7f3e1bc",
    "27446185-27c3-4c5d-b178-debf0401be69",
    "54964f95-2780-4c66-b9ef-43e8443d1335",
    "9fd1440b-d970-438c-a31d-1b68807111df",
    "1058bf2f-507b-409e-b072-e36c65f5415d",
]

OUT_DIR = Path(r"C:/Users/emman/OneDrive/Desktop/RCODES")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = OUT_DIR / "era5_macehead_20230214_1600.nc"
TMP_TARGET = OUT_DIR / "era5_macehead_20230214_1600.tmp.nc"

DATASET = "reanalysis-era5-pressure-levels"

REQUEST = {
    "product_type": "reanalysis",
    "variable": [
        "temperature",
        "specific_humidity",
        "u_component_of_wind",
        "v_component_of_wind",
    ],
    "pressure_level": [str(x) for x in range(1000, 75, -25)],
    "year": "2023",
    "month": "02",
    "day": "14",
    "time": "16:00",
    "area": [53.25, -9.50, 53.25, -9.50],
    "data_format": "netcdf_legacy",
}

disabled_keys = set()

def stamp(msg):
    print(f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {msg}", flush=True)

def valid_file(path):
    return path.exists() and path.stat().st_size > 0

def classify_error(e):
    s = str(e).lower()

    if any(x in s for x in [
        "401",
        "authentication required",
        "permission denied",
        "unauthorized",
        "forbidden",
        "invalid token",
        "invalid key",
        "disabled",
        "revoked",
    ]):
        return "auth"

    if any(x in s for x in [
        "number queued requests",
        "temporarily limited",
        "configure your scripts",
        "too many requests",
        "queued requests",
        "queue",
        "rate limit",
    ]):
        return "queue"

    return "other"

if valid_file(TARGET):
    stamp(f"Already downloaded: {TARGET}")
    raise SystemExit

attempt = 0

while True:
    usable_keys = [k for k in CDS_KEYS if k not in disabled_keys]

    if not usable_keys:
        raise RuntimeError("All supplied CDS tokens failed authentication or were disabled.")

    random.shuffle(usable_keys)

    for key in usable_keys:
        attempt += 1
        key_i = CDS_KEYS.index(key) + 1

        if TMP_TARGET.exists():
            TMP_TARGET.unlink()

        stamp(f"Attempt {attempt} | token {key_i}/{len(CDS_KEYS)} | submitting one request")

        try:
            client = cdsapi.Client(
                url="https://cds.climate.copernicus.eu/api",
                key=key,
                timeout=300,
                quiet=False,
                debug=False,
            )

            client.retrieve(DATASET, REQUEST, str(TMP_TARGET))

            if valid_file(TMP_TARGET):
                os.replace(TMP_TARGET, TARGET)
                stamp(f"Downloaded: {TARGET}")
                raise SystemExit

            stamp("CDS returned without a usable NetCDF file")

        except SystemExit:
            raise

        except Exception as e:
            kind = classify_error(e)

            if kind == "auth":
                disabled_keys.add(key)
                stamp(f"Token {key_i} failed authentication or is disabled; skipping it for this run.")

            elif kind == "queue":
                stamp("CDS dataset queue rejected the request.")

            else:
                stamp("Unexpected error:")
                stamp(str(e))

        delay = random.randint(2, 10)
        stamp(f"Waiting {delay} seconds before next attempt.")
        time.sleep(delay)

2026-05-25 10:48:00 | Attempt 1 | token 15/15 | submitting one request
2026-05-25 10:48:01 | Token 15 failed authentication or is disabled; skipping it for this run.
2026-05-25 10:48:01 | Waiting 8 seconds before next attempt.
2026-05-25 10:48:09 | Attempt 2 | token 2/15 | submitting one request
2026-05-25 10:48:10 | Token 2 failed authentication or is disabled; skipping it for this run.
2026-05-25 10:48:10 | Waiting 7 seconds before next attempt.
2026-05-25 10:48:17 | Attempt 3 | token 4/15 | submitting one request
2026-05-25 10:48:17 | Token 4 failed authentication or is disabled; skipping it for this run.
2026-05-25 10:48:17 | Waiting 3 seconds before next attempt.
2026-05-25 10:48:20 | Attempt 4 | token 6/15 | submitting one request
2026-05-25 10:48:23 | Token 6 failed authentication or is disabled; skipping it for this run.
2026-05-25 10:48:23 | Waiting 10 seconds before next attempt.
2026-05-25 10:48:33 | Attempt 5 | token 14/15 | submitting one request


2026-05-25 10:48:33,921 INFO Request ID is 094824ac-3a78-4a23-9f16-77a4ea4a077f
2026-05-25 10:48:33,995 INFO status has been updated to accepted
2026-05-25 10:48:47,613 INFO status has been updated to running
2026-05-25 10:48:55,318 INFO The 'netcdf_legacy' format is deprecated and no longer supported. Users are encouraged to update workflows to use the updated, and CF compliant, 'netcdf' option.
2026-05-25 10:48:55,318 INFO status has been updated to successful
                                                                                         

2026-05-25 10:48:56 | Downloaded: C:\Users\emman\OneDrive\Desktop\RCODES\era5_macehead_20230214_1600.nc


SystemExit: 